# FoodSafe-IoT: Retrain MobileNetV2 (v2 - Improved)
**Fixes potato/tomato misclassification as apple**

### What changed from v1:
1. **Class-weighted loss** - penalizes mistakes on underrepresented classes (potato has 4x fewer images than apple)
2. **Stronger data augmentation** - forces the model to learn texture/shape, not just color
3. **Partial backbone unfreezing** - unfreezes last 4 conv blocks so model learns food-specific low-level features
4. **15 epochs** instead of 10
5. **Confusion matrix** to visually verify apple/potato/tomato are no longer confused

**Before running:**
1. Go to `Runtime` -> `Change runtime type` -> Select **T4 GPU**
2. Run cells one by one

## Step 0: Verify GPU is Available

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found! Go to Runtime -> Change runtime type -> T4 GPU")

## Step 1: Setup Kaggle & Download Dataset

**Paste your Kaggle username and API token below.**

In [ ]:
import os

# PASTE YOUR USERNAME AND TOKEN HERE
os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME_HERE' 
os.environ['KAGGLE_KEY'] = 'YOUR_COPIED_TOKEN_HERE'

# Install Kaggle CLI
!pip install -q kaggle
print("Kaggle API configured with your token!")

In [ ]:
# Download the dataset directly inside Colab
!kaggle datasets download -d swoyam2609/fresh-and-stale-classification -p /content/data --unzip
print("\nDataset downloaded and extracted!")

In [ ]:
import os

# Find the dataset folders
DATASET_ROOT = '/content/data'
for root, dirs, _files in os.walk(DATASET_ROOT):
    if 'Train' in dirs and 'Test' in dirs:
        DATASET_ROOT = root
        break

train_dir = os.path.join(DATASET_ROOT, 'Train')
test_dir = os.path.join(DATASET_ROOT, 'Test')

# Count images per class
print("Dataset root:", DATASET_ROOT)
print("\nImages per class (Train):")
total_train = 0
for cls in sorted(os.listdir(train_dir)):
    count = len(os.listdir(os.path.join(train_dir, cls)))
    total_train += count
    print(f"  {cls}: {count}")

total_test = sum(len(f) for _, _, f in os.walk(test_dir))
print(f"\nTotal Train: {total_train} | Total Test: {total_test} | Grand Total: {total_train + total_test}")

## Step 2: Prepare Data Loaders (IMPROVED Augmentation)

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np

# IMPROVED: Much stronger data augmentation
# - RandomPerspective: forces model to learn 3D shape, not just flat color blobs
# - Stronger ColorJitter: prevents model from relying purely on color (red = apple)
# - GaussianBlur: forces model to learn texture at different sharpness levels
# - RandomAffine: slight scaling/translation for robustness
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(25),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets using ImageFolder (folder name = class label)
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)

# Split training into train + validation
val_size = int(0.15 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size],
                                        generator=torch.Generator().manual_seed(42))

# Create data loaders
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Print class info
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Dataset loaded!")
print(f"  Train: {train_size} images")
print(f"  Val:   {val_size} images")
print(f"  Test:  {len(test_dataset)} images")
print(f"  Classes ({num_classes}): {class_names}")

# ---- IMPROVED: Compute class weights to handle imbalance ----
# Count images per class
class_counts = np.zeros(num_classes)
for _, label in train_dataset.samples:
    class_counts[label] += 1

# Inverse frequency weighting: rare classes get higher weight
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * num_classes  # normalize
class_weights = torch.FloatTensor(class_weights)

print(f"\nClass weights (higher = rarer class):")
for name, weight in zip(class_names, class_weights):
    bar = '#' * int(weight * 5)
    print(f"  {name:20s}: {weight:.3f} {bar}")

## Step 3: Build MobileNetV2 Model (IMPROVED - Partial Unfreezing)

In [ ]:
import torchvision.models as models
from torch import nn

# Load pre-trained MobileNetV2 (trained on 14 million ImageNet images)
model = models.mobilenet_v2(weights='IMAGENET1K_V1')

# IMPROVED: Freeze only the early layers, unfreeze the last 4 feature blocks
# Early layers learn generic features (edges, colors) - keep them frozen
# Later layers learn high-level features - we want these to learn food-specific patterns
# like potato skin roughness vs apple glossiness
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze last 4 inverted residual blocks (features[15], [16], [17], [18])
for block in model.features[-4:]:
    for param in block.parameters():
        param.requires_grad = True

# Replace the classifier head for our 18 food classes
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, num_classes)
)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"MobileNetV2 loaded on {device}")
print(f"  Output classes: {num_classes}")

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"  (v1 only trained ~1.5% - now training {100*trainable/total:.1f}% for better food-specific features)")

## Step 4: Train the Model (IMPROVED - Weighted Loss + 15 Epochs)

In [ ]:
from torch import optim
import time

# IMPROVED: Use class-weighted loss so rare classes (potato) matter more
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# IMPROVED: Use different learning rates for backbone vs classifier
# Backbone (unfrozen layers) gets a smaller LR to not destroy pre-trained features
optimizer = optim.Adam([
    {'params': model.features[-4:].parameters(), 'lr': 0.0001},  # backbone: careful fine-tuning
    {'params': model.classifier.parameters(), 'lr': 0.001},       # classifier: faster learning
])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

# IMPROVED: 15 epochs instead of 10
NUM_EPOCHS = 15
best_val_acc = 0.0

print(f"Training for {NUM_EPOCHS} epochs on {device}...")
print(f"Using class-weighted loss to fix apple/potato/tomato confusion\n")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # ---- TRAIN ----
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    train_acc = 100. * train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)
    
    # ---- VALIDATE ----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler.step(avg_val_loss)
    epoch_time = time.time() - epoch_start
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({epoch_time:.0f}s) | "
          f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.1f}% | "
          f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.1f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': class_names,
            'num_classes': num_classes,
            'best_val_acc': best_val_acc
        }, '/content/food_spoilage_model_best.pth')
        print(f"        >> Best model saved! (Val Acc: {best_val_acc:.1f}%)")

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time/60:.1f} minutes!")
print(f"Best validation accuracy: {best_val_acc:.1f}%")

## Step 5: Test Final Accuracy + Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
checkpoint = torch.load('/content/food_spoilage_model_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Overall accuracy
test_acc = 100. * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
print(f"TEST SET RESULTS")
print(f"  Overall Accuracy: {test_acc:.1f}%")
print(f"  Total test images: {len(all_labels)}")

# Per-class report
print(f"\nPer-Class Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Fresh vs Rotten binary accuracy
fresh_classes = [i for i, c in enumerate(class_names) if 'fresh' in c.lower()]
rotten_classes = [i for i, c in enumerate(class_names) if 'rotten' in c.lower()]

binary_preds = ['fresh' if p in fresh_classes else 'rotten' for p in all_preds]
binary_labels = ['fresh' if l in fresh_classes else 'rotten' for l in all_labels]
binary_acc = 100. * sum(p == l for p, l in zip(binary_preds, binary_labels)) / len(binary_labels)

print(f"Fresh vs Rotten Binary Accuracy: {binary_acc:.1f}%")

In [ ]:
# IMPROVED: Confusion Matrix to visually verify apple/potato/tomato separation
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix - FoodSafe-IoT v2\n(Check apple/potato/tomato rows for improvements)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Highlight the key classes we wanted to fix
print("\n=== KEY CLASSES TO CHECK ===")
focus_classes = ['freshapples', 'freshpotato', 'freshtomato',
                 'rottenapples', 'rottenpotato', 'rottentomato']
for cls_name in focus_classes:
    if cls_name in class_names:
        idx = class_names.index(cls_name)
        total = cm[idx].sum()
        correct = cm[idx][idx]
        acc = 100.0 * correct / total if total > 0 else 0
        misclass = [(class_names[j], cm[idx][j]) for j in range(len(class_names)) if j != idx and cm[idx][j] > 0]
        misclass.sort(key=lambda x: -x[1])
        print(f"\n  {cls_name}: {acc:.1f}% correct ({correct}/{total})")
        if misclass:
            top_mistakes = misclass[:3]
            print(f"    Top mistakes: {', '.join(f'{name}({count})' for name, count in top_mistakes)}")

## Step 6: Save Model to Google Drive & Download

In [ ]:
from google.colab import drive, files
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Save to Google Drive
drive_path = '/content/drive/MyDrive/food_spoilage_model_best.pth'
shutil.copy('/content/food_spoilage_model_best.pth', drive_path)

print(f"Model saved to Google Drive!")
print(f"  Path: {drive_path}")
print(f"  Size: {os.path.getsize(drive_path) / (1024*1024):.1f} MB")
print(f"  Classes: {class_names}")
print(f"  Test Accuracy: {test_acc:.1f}%")
print(f"\nAlso downloading to your browser now...")

# Also trigger a direct browser download
files.download('/content/food_spoilage_model_best.pth')

## Step 7: Quick Test - Predict on a Single Image

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import random

# Pick a random test image
test_classes = os.listdir(test_dir)
random_class = random.choice(test_classes)
class_dir = os.path.join(test_dir, random_class)
random_img = random.choice(os.listdir(class_dir))
img_path = os.path.join(class_dir, random_img)

# Load and predict
img = Image.open(img_path).convert('RGB')
img_tensor = test_transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(img_tensor)
    probs = torch.nn.functional.softmax(output, dim=1)
    confidence, predicted = probs.max(1)

pred_class = class_names[predicted.item()]
is_fresh = 'fresh' in pred_class.lower()
status = 'FRESH' if is_fresh else 'ROTTEN'

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title(f"Prediction: {pred_class}\n{status} ({confidence.item():.1%})\nActual: {random_class}", fontsize=14)
plt.axis('off')
plt.show()

print(f"Prediction: {pred_class} ({confidence.item():.1%})")
print(f"Actual: {random_class}")
print(f"Status: {status}")

---
## Done! Next Steps:
1. The model file `food_spoilage_model_best.pth` has been downloaded to your browser
2. Move it to: `D:\FoodSafe_IoT\02_models\food_spoilage_model_best.pth` (replace the old one)
3. Restart the Flask server on your laptop
4. Test with potato and tomato images - they should now be correctly classified!